# RAG Pipeline & Vector Database
In this notebook, we will:
1. Generate synthetic hospital policy documents (Admission, Billing, Discharge, Insurance, Emergency).
2. Chunk the documents into smaller, manageable pieces.
3. Generate embeddings using a free, local HuggingFace model.
4. Store the embeddings in a FAISS vector database.
5. Test the retrieval pipeline to ensure it accurately finds the right policy context.

# Setup & Create Synthetic Policies

In [2]:
import os

# Define paths (since this notebook is in the 'notebooks' folder, we go up one level '..')
base_dir = '..'
policies_dir = os.path.join(base_dir, 'policies')
faiss_dir = os.path.join(base_dir, 'faiss_index')

# Create the policies directory
os.makedirs(policies_dir, exist_ok=True)

# Define the synthetic hospital policies
policies = {
    "admission_policy.txt": """Hospital Admission Policy: All patients must present a valid government-issued ID and their active insurance card upon arrival. Emergency admissions are triaged and processed within 15 minutes of arrival. Elective admissions require prior scheduling at least 48 hours in advance and a completed pre-admission health check.""",
    
    "billing_policy.txt": """Billing and Payment Policy: Final billing statements are generated immediately upon patient discharge. Patients have exactly 30 days to settle their outstanding bills in full. Unpaid bills after 60 days will be automatically sent to third-party collections. Insurance claims are processed and submitted within 14 days of discharge.""",
    
    "discharge_policy.txt": """Discharge Policy: Patients can only be officially discharged when the attending doctor signs the final discharge summary. All prescribed medications and follow-up outpatient appointments must be arranged and confirmed before the patient leaves the premises. The average administrative discharge processing time is 2 hours.""",
    
    "insurance_policy.txt": """Insurance Approval Policy: Prior approval from the insurance provider is strictly required for all surgical procedures and advanced imaging (MRI/CT scans). Routine consultations, basic blood tests, and X-rays do not require prior approval. Out-of-network services may incur a mandatory 20% co-pay for the patient.""",
    
    "emergency_policy.txt": """Emergency Room Policy: The Emergency Room (ER) operates 24/7, 365 days a year. Medical triage is strictly based on the severity of the condition, not the time of arrival. Critical patients are seen immediately by a trauma team. Non-critical patients may wait up to 4 hours. To maintain a sterile environment, visitors are strictly limited to a maximum of 2 per patient in the ER."""
}

# Save the policies as individual text files
for filename, content in policies.items():
    file_path = os.path.join(policies_dir, filename)
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(content)

print(f"✅ Successfully created {len(policies)} synthetic policy documents in the '{policies_dir}' directory.")

✅ Successfully created 5 synthetic policy documents in the '..\policies' directory.


# Load, Chunk, and Embed

### Import 

In [4]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("Loading documents and generating embeddings (this may take a minute the first time)...")

Loading documents and generating embeddings (this may take a minute the first time)...


In [5]:
# 1. Load all text files from the policies directory
loader = DirectoryLoader(policies_dir, glob="**/*.txt", loader_cls=TextLoader)
docs = loader.load()
print(f"Loaded {len(docs)} raw documents.")

Loaded 5 raw documents.


In [6]:
# 2. Chunk the documents (crucial for RAG so the LLM doesn't get overwhelmed)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(docs)
print(f"Split documents into {len(chunks)} smaller chunks.")

Split documents into 5 smaller chunks.


In [8]:
# 3. Generate embeddings using a FREE local HuggingFace model
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6303.63it/s]


In [11]:
# 4. Create the FAISS vector database and save it to disk
vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local(faiss_dir)
print(f"✅ FAISS vector database successfully saved to '{faiss_dir}'!")

✅ FAISS vector database successfully saved to '..\faiss_index'!


# Test the Retrieval Pipeline

In [12]:
print("Testing the RAG Retrieval Pipeline...")

# Load the saved FAISS database
loaded_vectorstore = FAISS.load_local(faiss_dir, embeddings, allow_dangerous_deserialization=True)

# Create a retriever that fetches the top 2 most relevant chunks
retriever = loaded_vectorstore.as_retriever(search_kwargs={"k": 2})

# Define a test query that a hospital staff member might ask
test_query = "What is the maximum number of visitors allowed in the ER?"

# Retrieve the most relevant context
retrieved_docs = retriever.invoke(test_query)

print(f"\n🔍 Test Query: '{test_query}'")
print("-" * 60)

Testing the RAG Retrieval Pipeline...

🔍 Test Query: 'What is the maximum number of visitors allowed in the ER?'
------------------------------------------------------------


In [13]:
for i, doc in enumerate(retrieved_docs):
    print(f"\n📄 Retrieved Context Chunk {i+1} (Source: {doc.metadata.get('source', 'Unknown')}):")
    print(doc.page_content)
    print("-" * 60)

print("\n✅ RAG Pipeline test complete! The vector search successfully found the correct policy.")


📄 Retrieved Context Chunk 1 (Source: ..\policies\emergency_policy.txt):
Emergency Room Policy: The Emergency Room (ER) operates 24/7, 365 days a year. Medical triage is strictly based on the severity of the condition, not the time of arrival. Critical patients are seen immediately by a trauma team. Non-critical patients may wait up to 4 hours. To maintain a sterile environment, visitors are strictly limited to a maximum of 2 per patient in the ER.
------------------------------------------------------------

📄 Retrieved Context Chunk 2 (Source: ..\policies\admission_policy.txt):
Hospital Admission Policy: All patients must present a valid government-issued ID and their active insurance card upon arrival. Emergency admissions are triaged and processed within 15 minutes of arrival. Elective admissions require prior scheduling at least 48 hours in advance and a completed pre-admission health check.
------------------------------------------------------------

✅ RAG Pipeline test complete